In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, KBinsDiscretizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

# Đọc file sạch từ bước EDA
df = pd.read_csv('../data/processed/german_clean.csv')
print("Shape:", df.shape)
print("\nCác cột:")
print(df.columns.tolist())

Shape: (1000, 35)

Các cột:
['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'residence_since', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone', 'foreign_worker', 'credit_risk', 'checking_status_label', 'credit_history_label', 'purpose_label', 'savings_label', 'employment_label', 'personal_status_label', 'other_parties_label', 'property_label', 'other_plans_label', 'housing_label', 'job_label', 'telephone_label', 'foreign_worker_label', 'credit_risk_label']


In [2]:
# Tách features và target
# Bỏ các cột _label (chỉ dùng để giải thích, không đưa vào mô hình)
# Bỏ credit_risk (target gốc số), dùng credit_risk_label
drop_cols = [c for c in df.columns if c.endswith('_label')] + ['credit_risk']
X = df.drop(columns=drop_cols)
y = df['credit_risk'].map({1: 1, 2: 0})  # 1 = Good, 0 = Bad (dương tính = Good)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nPhân bố y:")
print(y.value_counts())
print("\nCác cột trong X:")
print(X.columns.tolist())
print("\nKiểu dữ liệu X:")
print(X.dtypes.value_counts())

X shape: (1000, 20)
y shape: (1000,)

Phân bố y:
credit_risk
1    700
0    300
Name: count, dtype: int64

Các cột trong X:
['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'residence_since', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone', 'foreign_worker']

Kiểu dữ liệu X:
str      13
int64     7
Name: count, dtype: int64


In [3]:
# Chia 60% train, 20% validation, 20% test
# stratify=y để giữ tỷ lệ 70/30 ở mỗi tập
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape, "| Good:", (y_train==1).sum(), "Bad:", (y_train==0).sum())
print("Val:  ", X_val.shape,   "| Good:", (y_val==1).sum(),   "Bad:", (y_val==0).sum())
print("Test: ", X_test.shape,  "| Good:", (y_test==1).sum(),  "Bad:", (y_test==0).sum())

Train: (600, 20) | Good: 420 Bad: 180
Val:   (200, 20) | Good: 140 Bad: 60
Test:  (200, 20) | Good: 140 Bad: 60


In [4]:
# Xác định biến số và biến hạng mục
numeric_features = ['duration', 'credit_amount', 'installment_commitment',
                    'residence_since', 'age', 'existing_credits', 'num_dependents']
categorical_features = ['checking_status', 'credit_history', 'purpose',
                        'savings_status', 'employment', 'personal_status',
                        'other_parties', 'property_magnitude',
                        'other_payment_plans', 'housing', 'job',
                        'own_telephone', 'foreign_worker']

# Pipeline cho biến số: StandardScaler
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Pipeline cho biến hạng mục: OneHotEncoder
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Kết hợp
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Đã định nghĩa preprocessor:")
print(f"  - Numeric: {len(numeric_features)} biến → StandardScaler")
print(f"  - Categorical: {len(categorical_features)} biến → OneHotEncoder")

Đã định nghĩa preprocessor:
  - Numeric: 7 biến → StandardScaler
  - Categorical: 13 biến → OneHotEncoder


In [5]:
# QUAN TRỌNG: fit CHỈ trên train, transform val/test
# Đây là cách chống data leakage
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Sau khi tiền xử lý:")
print("  Train:", X_train_processed.shape)
print("  Val:  ", X_val_processed.shape)
print("  Test: ", X_test_processed.shape)

# Lấy tên cột sau OneHot
feature_names = (
    numeric_features +
    list(preprocessor.named_transformers_['cat']
         .named_steps['onehot']
         .get_feature_names_out(categorical_features))
)
print(f"\nTổng số features sau OneHot: {len(feature_names)}")

Sau khi tiền xử lý:
  Train: (600, 61)
  Val:   (200, 61)
  Test:  (200, 61)

Tổng số features sau OneHot: 61


In [6]:
# Chuyển về DataFrame để dễ đọc
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_val_df = pd.DataFrame(X_val_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

# Thêm target
X_train_df['target'] = y_train.values
X_val_df['target'] = y_val.values
X_test_df['target'] = y_test.values

# Lưu
X_train_df.to_csv('../data/processed/train.csv', index=False)
X_val_df.to_csv('../data/processed/val.csv', index=False)
X_test_df.to_csv('../data/processed/test.csv', index=False)

# Lưu preprocessor để dùng lại sau
joblib.dump(preprocessor, '../data/processed/preprocessor.pkl')

print("Đã lưu:")
print("  train.csv:", X_train_df.shape)
print("  val.csv:  ", X_val_df.shape)
print("  test.csv: ", X_test_df.shape)
print("  preprocessor.pkl")

Đã lưu:
  train.csv: (600, 62)
  val.csv:   (200, 62)
  test.csv:  (200, 62)
  preprocessor.pkl


In [7]:
# Kiểm tra không có NaN
print("NaN trong train:", X_train_df.isna().sum().sum())
print("NaN trong val:", X_val_df.isna().sum().sum())
print("NaN trong test:", X_test_df.isna().sum().sum())

# Kiểm tra phân bố target
print("\nPhân bố target:")
print("  Train:", X_train_df['target'].value_counts().to_dict())
print("  Val:  ", X_val_df['target'].value_counts().to_dict())
print("  Test: ", X_test_df['target'].value_counts().to_dict())

NaN trong train: 0
NaN trong val: 0
NaN trong test: 0

Phân bố target:
  Train: {1: 420, 0: 180}
  Val:   {1: 140, 0: 60}
  Test:  {1: 140, 0: 60}
